# Notebook 02: TTS PoC (Indic Parler-TTS)

This notebook verifies speech synthesis using `ai4bharat/indic-parler-tts`.
It tests Odia text synthesis, dual tokenizer setup, voice descriptions, and audio playback.

In [ ]:
# Install dependencies (run once in Colab/Kaggle)
!pip install -q transformers soundfile accelerate
!pip install -q git+https://github.com/huggingface/parler-tts.git

In [ ]:
import os
import torch
import soundfile as sf
from IPython.display import Audio, display
from transformers import AutoTokenizer
from parler_tts import ParlerTTSForConditionalGeneration

# Set your Hugging Face access token (required for gated access)
# os.environ["HF_TOKEN"] = "hf_..."

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if "cuda" in DEVICE else torch.float32
TTS_MODEL_ID = "ai4bharat/indic-parler-tts"

print(f"Device: {DEVICE} ({DTYPE})")

In [ ]:
# Load Parler-TTS and dual tokenizers
print("Loading Indic Parler-TTS model...")
model = ParlerTTSForConditionalGeneration.from_pretrained(
    TTS_MODEL_ID,
    torch_dtype=DTYPE
).to(DEVICE)
model.eval()

sampling_rate = model.config.sampling_rate
print(f"Model loaded. Native sampling rate: {sampling_rate} Hz")

tokenizer = AutoTokenizer.from_pretrained(TTS_MODEL_ID)
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
print("Tokenizers initialized.")

In [ ]:
def speak(text, description):
    """Synthesize speech from prompt text and voice description."""
    prompt_ids = tokenizer(text, return_tensors="pt").input_ids.to(DEVICE)
    desc_ids = description_tokenizer(description, return_tensors="pt").input_ids.to(DEVICE)
    
    with torch.inference_mode():
        out = model.generate(input_ids=desc_ids, prompt_input_ids=prompt_ids)
        
    audio = out.cpu().numpy().squeeze()
    return audio

# Hand-written Odia test sentence
odia_prompt = "ଓଡ଼ିଶା କୁ ସ୍ୱାଗତ । ଭୁବନେଶ୍ୱର ହେଉଛି ମନ୍ଦିର ମାଳିନୀ ନଗରୀ ।"

# Test 1: Female voice preset
desc_female = "A female speaker speaks in a clear, natural, moderately paced voice with close recording and no background noise."
print("Generating Odia speech (Female preset)...")
audio_female = speak(odia_prompt, desc_female)
display(Audio(audio_female, rate=sampling_rate))

# Test 2: Male voice preset
desc_male = "A male speaker speaks in a clear, natural, resonant, moderately paced voice with close recording and no background noise."
print("Generating Odia speech (Male preset)...")
audio_male = speak(odia_prompt, desc_male)
display(Audio(audio_male, rate=sampling_rate))